In [1]:
import os, sys, time
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")

from langchain_community.utilities import SQLDatabase

from src.chains import build_llm, SMALL_MODEL
from src.tools import search_video, search_wikipedia, search_technique

llm = build_llm()  # gpt-oss-120b
llm_small = build_llm(SMALL_MODEL)  # gpt-oss-20b: gerar SQL / reformular queries

# read-only na conexão = 1ª camada de segurança do text-to-SQL
db = SQLDatabase.from_uri("sqlite:///file:../data/db/bjj.sqlite?mode=ro&uri=true")
print(db.dialect, db.get_usable_table_names())
print(db.get_table_info())

print(
    db.run(
        "SELECT submission, COUNT(*) AS n FROM lutas "
        "WHERE submission IS NOT NULL GROUP BY submission ORDER BY n DESC LIMIT 5"
    )
)

try:
    db.run("DELETE FROM lutas WHERE 1=0")
except Exception as e:
    print("read-only OK:", type(e).__name__, str(e)[:80])

C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_21220\4271154107.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


sqlite ['lutas']

CREATE TABLE lutas (
	victory_method TEXT, 
	submission TEXT, 
	winner_points INTEGER, 
	loser_points INTEGER, 
	female INTEGER, 
	year INTEGER, 
	absolute INTEGER, 
	weight_class REAL, 
	importance INTEGER, 
	total_points INTEGER, 
	submission_target TEXT, 
	winner_name TEXT, 
	loser_name TEXT
)

/*
3 rows from lutas table:
victory_method	submission	winner_points	loser_points	female	year	absolute	weight_class	importance	total_points	submission_target	winner_name	loser_name
DECISION	None	-1	-1	0	2011	1	None	1	2	None	Murilo Santana	Vinicius Magalhaes
SUBMISSION	Kimura	-1	-1	0	2022	0	3.0	0	2	Arm	Nicholas Meregali	Henrique Cardoso
DECISION	None	-1	-1	0	2022	0	3.0	1	2	None	Nicholas Meregali	Yuri Simoes
*/
[('RNC', 99), ('Armbar', 68), ('Guillotine', 34), ('Heel hook', 32), ('Triangle', 24)]
read-only OK: OperationalError (sqlite3.OperationalError) attempt to write a readonly database
[SQL: DELETE FRO


In [2]:
import importlib, src.tools
importlib.reload(src.tools)          # src.tools já foi importado na célula 1
from src.tools import ibjjf_points

print(ibjjf_points.invoke("Guard pass"))
print(ibjjf_points.invoke("berimbolo"))
print(ibjjf_points.name, "|", ibjjf_points.args)


guard pass: 3 points (IBJJF)
Unknown action 'berimbolo'. Valid: takedown, sweep, knee on belly, guard pass, mount, back control
ibjjf_points | {'action': {'title': 'Action', 'type': 'string'}}


## 4.1 Tipologia das ferramentas

| Tipo            | Tool                               | Quando usar                          | Custo                 | Determinismo                      | Risco                                             |
| --------------- | ---------------------------------- | ------------------------------------ | --------------------- | --------------------------------- | ------------------------------------------------- |
| Hardcoded       | `ibjjf_points`                     | regra fixa e pequena (pontuação)     | zero, sem rede        | total                             | desatualizar se a regra mudar                     |
| Semântica (RAG) | `search_technique`                 | "como fazer/defender X" na KB curada | embedding local (CPU) | alto (mesmo índice → mesmo top-k) | chunk errado se o vocabulário não casar (ver 4.5) |
| API externa     | `search_video`, `search_wikipedia` | vídeo / fato histórico fora da KB    | rede + quota          | baixo (resultado muda no tempo)   | timeout, quota, conteúdo não curado               |
| Text-to-query   | `query_adcc` (4.3)                 | estatística sobre dados estruturados | 1 chamada LLM + SQL   | médio (LLM gera o SQL)            | SQL errado ou perigoso → read-only + validação    |


In [3]:
importlib.reload(src.tools)
from src.tools import search_wikipedia, search_video, _wiki_lookup

for i in range(2):
    t0 = time.perf_counter()
    out = search_wikipedia.invoke("Helio Gracie")
    print(f"call {i+1}: {time.perf_counter()-t0:.3f}s | {out[:60]}")

print(_wiki_lookup.cache_info())
print(search_video.invoke("kimura closed guard bjj"))


call 1: 0.678s | Hélio Gracie: Hélio Gracie was a Brazilian martial artist wh
call 2: 0.002s | Hélio Gracie: Hélio Gracie was a Brazilian martial artist wh
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)
Kimura Tutorial — https://www.youtube.com/watch?v=fqQ4mVxJaoE


## 4.2 Integração de APIs por tipo de uso

| Uso | Tool | Fonte | Timeout | Erro | Retry | Cache | Dummy |
|---|---|---|---|---|---|---|---|
| Busca (descoberta) | `search_video` | YouTube Data API v3 (quota 10k/dia, 100 por busca) | 10s | string → agente segue | 2x em 429/5xx | `lru_cache` 128 | sim |
| Consulta (lookup factual) | `search_wikipedia` | Wikipedia REST (sem chave, exige User-Agent) | 10s | string | 2x em 429/5xx | `lru_cache` 128 | sim |
| Local (conhecimento curado) | `search_technique` | FAISS em disco | — | string | — | índice carregado 1x | sim |
| Local (dados estruturados) | `query_adcc` | SQLite read-only | — | string + retry c/ erro no prompt (4.3) | — | — | sim |
| Ação/escrita | nenhuma | — | — | — | — | — | — |

Sem tools de escrita por decisão: o agente é tutor, não executa ações no mundo (agendar, postar, pagar). Isso elimina a necessidade de confirmação humana / idempotência nesta fase.


## 4.3 Text-to-SQL sobre SQLite

Pergunta em linguagem natural → LLM gera SQL estruturado (`SQLQuery`) → validação → execução read-only → resposta.
Segurança em 2 camadas: (1) conexão `mode=ro` (provada na célula 1); (2) validador que só aceita um único SELECT.


In [4]:
import re
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


class SQLQuery(BaseModel):
    reasoning: str = Field(
        description="Short reasoning: which columns and filters answer the question"
    )
    sql: str = Field(description="One SQLite SELECT query")


sql_parser = PydanticOutputParser(pydantic_object=SQLQuery)

COLUMN_DICT = """Column notes (table lutas = one ADCC match per row, 1998-2022):
- victory_method: 'SUBMISSION', 'POINTS', 'DECISION', 'INJURY', 'DESQUALIFICATION' (sic, misspelled)
- submission: free text, only when victory_method='SUBMISSION', else NULL. Ex: 'RNC' (rear naked choke),
  'Armbar', 'Guillotine', 'Heel hook', 'Inside heel hook', 'Outside heel hook', 'Triangle', 'Kimura'.
  Use LIKE '%heel hook%' to get all heel hook variants.
- submission_target: 'Arm', 'Leg', 'Neck', 'Other/Unknown', or NULL (not a submission)
- winner_points / loser_points: -1 means NO DATA (exclude with > -1 before averaging)
- female: 1 = women's division, 0 = men's
- absolute: 1 = open weight (absolute) division; then weight_class is NULL
- weight_class: code 0-4 (lightest to heaviest), NULL when absolute
- importance: round importance code 0-5 (higher = later round)
- winner_name / loser_name: athlete full name"""

FEW_SHOT = """Examples:
Q: Most common submission ever?
SQL: SELECT submission, COUNT(*) AS n FROM lutas WHERE submission IS NOT NULL GROUP BY submission ORDER BY n DESC LIMIT 1
Q: How many women's matches ended by submission?
SQL: SELECT COUNT(*) FROM lutas WHERE female = 1 AND victory_method = 'SUBMISSION' LIMIT 1
Q: Average winner points in 2019, when available?
SQL: SELECT AVG(winner_points) FROM lutas WHERE year = 2019 AND winner_points > -1 LIMIT 1"""

sql_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You write SQLite queries for an ADCC (submission grappling) match database.\n"
            "Schema:\n{schema}\n\n{column_dict}\n\n{few_shot}\n\n"
            "Rules: a single SELECT only (no ;, no writes). Always end the query with a LIMIT clause "
            "(LIMIT 20, or LIMIT 1 for a single aggregate). "
            "Use only the columns above. Athlete names: use LIKE '%name%'.\n\n{format_instructions}",
        ),
        ("human", "{question}"),
    ]
).partial(
    schema=db.get_table_info(),
    column_dict=COLUMN_DICT,
    few_shot=FEW_SHOT,
    format_instructions=sql_parser.get_format_instructions(),
)

sql_gen_chain = sql_prompt | llm_small | sql_parser

FORBIDDEN = re.compile(
    r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum)\b", re.I
)


def validate_sql(sql: str) -> str:
    """2ª camada de segurança: só um SELECT, sem comandos de escrita."""
    s = sql.strip().rstrip(";").strip()
    if not s.lower().startswith(("select", "with")):
        raise ValueError("only SELECT queries are allowed")
    if ";" in s:
        raise ValueError("multiple statements are not allowed")
    if FORBIDDEN.search(s):
        raise ValueError(f"forbidden keyword: {FORBIDDEN.search(s).group()}")
    return s


# guarda: casos maliciosos (sem LLM)
for bad in [
    "DROP TABLE lutas",
    "SELECT 1; DELETE FROM lutas",
    "PRAGMA table_info(lutas)",
]:
    try:
        validate_sql(bad)
    except ValueError as e:
        print(f"BLOCKED  {bad!r:35} -> {e}")

# geração: 3 perguntas
for q in [
    "Quantas lutas terminaram em finalização em 2022?",
    "Qual atleta tem mais vitórias?",
    "Quantas vitórias por heel hook existem, somando todas as variações?",
]:
    out = sql_gen_chain.invoke({"question": q})
    print(
        f"\n>>> {q}\n    reasoning: {out.reasoning}\n    sql: {validate_sql(out.sql)}"
    )
    time.sleep(5)

BLOCKED  'DROP TABLE lutas'                  -> only SELECT queries are allowed
BLOCKED  'SELECT 1; DELETE FROM lutas'       -> multiple statements are not allowed
BLOCKED  'PRAGMA table_info(lutas)'          -> only SELECT queries are allowed

>>> Quantas lutas terminaram em finalização em 2022?
    reasoning: Use the victory_method column to filter for 'SUBMISSION' and the year column to restrict to 2022, then count the matching rows.
    sql: SELECT COUNT(*) FROM lutas WHERE year = 2022 AND victory_method = 'SUBMISSION' LIMIT 1

>>> Qual atleta tem mais vitórias?
    reasoning: To find the athlete with the most victories, we count how many times each name appears in the winner_name column, group by that name, order by the count descending, and return the top result.
    sql: SELECT winner_name, COUNT(*) AS n FROM lutas GROUP BY winner_name ORDER BY n DESC LIMIT 1

>>> Quantas vitórias por heel hook existem, somando todas as variações?
    reasoning: We need to count all matches wher

In [5]:
import sqlite3
from langchain_core.output_parsers import StrOutputParser


def run_sql_question(question: str, max_attempts: int = 2) -> dict:
    """Gera SQL, valida e executa. Se falhar, devolve o erro ao LLM e tenta de novo."""
    prompt_q, sql, error = question, "", ""
    for attempt in range(1, max_attempts + 1):
        out = sql_gen_chain.invoke({"question": prompt_q})
        sql = out.sql
        try:
            rows = db.run(validate_sql(sql))
            return {"sql": sql, "rows": rows, "attempts": attempt, "error": None}
        except Exception as e:
            error = str(e).split("\n")[0][:200]  # só a 1ª linha do erro
            prompt_q = (
                f"{question}\n\nYour previous query failed.\n"
                f"Query: {sql}\nError: {error}\nWrite a corrected query."
            )
    return {"sql": sql, "rows": None, "attempts": attempt, "error": error}


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você responde perguntas sobre estatísticas do ADCC. Use SOMENTE o resultado SQL abaixo. "
            "Responda em português, em 1-2 frases. Se o resultado estiver vazio, diga que não há dados.\n\n"
            "SQL: {sql}\nResultado: {rows}",
        ),
        ("human", "{question}"),
    ]
)
answer_chain = answer_prompt | llm_small | StrOutputParser()


# gabarito calculado direto no SQLite (sem LLM)
raw = sqlite3.connect("../data/db/bjj.sqlite")
gold_sql = {
    "Quantas lutas terminaram em finalização em 2022?": "SELECT COUNT(*) FROM lutas WHERE year=2022 AND victory_method='SUBMISSION'",
    "Qual atleta tem mais vitórias?": "SELECT winner_name FROM lutas GROUP BY winner_name ORDER BY COUNT(*) DESC LIMIT 1",
    "Quantas vitórias por heel hook existem, somando todas as variações?": "SELECT COUNT(*) FROM lutas WHERE submission LIKE '%heel hook%'",
    "Qual a finalização mais comum da história do ADCC?": "SELECT submission FROM lutas WHERE submission IS NOT NULL GROUP BY submission ORDER BY COUNT(*) DESC LIMIT 1",
    "Quantas vitórias o Marcelo Garcia tem?": "SELECT COUNT(*) FROM lutas WHERE winner_name LIKE '%Marcelo Garcia%'",
    "Qual a média de pontos do vencedor, considerando só lutas com placar registrado?": "SELECT ROUND(AVG(winner_points), 2) FROM lutas WHERE winner_points > -1",
}


def matches(gold: str, rows) -> bool:
    """Número: compara com tolerância. Texto: substring."""
    if rows is None:
        return False
    try:
        g = float(gold)
    except ValueError:
        return gold in rows
    nums = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", rows)]
    return any(abs(n - g) < 0.01 for n in nums)


results = []
for q, gsql in gold_sql.items():
    gold = str(raw.execute(gsql).fetchone()[0])
    r = run_sql_question(q)
    ok = matches(gold, r["rows"])
    results.append(
        {
            "question": q,
            "gold": gold,
            "ok": ok,
            "attempts": r["attempts"],
            "sql": r["sql"],
        }
    )
    print(
        f"{'OK ' if ok else 'ERR'} [{r['attempts']}x] {q}\n     gold={gold} | rows={r['rows']} | err={r['error']}"
    )
    time.sleep(5)

print(f"\nAcerto: {sum(x['ok'] for x in results)}/{len(results)}")

# resposta final em linguagem natural (1 exemplo)
r = run_sql_question("Quantas vitórias o Marcelo Garcia tem?")
print(
    "\n",
    answer_chain.invoke({"question": "Quantas vitórias o Marcelo Garcia tem?", **r}),
)

OK  [1x] Quantas lutas terminaram em finalização em 2022?
     gold=40 | rows=[(40,)] | err=None
OK  [1x] Qual atleta tem mais vitórias?
     gold=Alexandre Ribeiro | rows=[('Alexandre Ribeiro', 30)] | err=None
OK  [1x] Quantas vitórias por heel hook existem, somando todas as variações?
     gold=52 | rows=[(52,)] | err=None
OK  [1x] Qual a finalização mais comum da história do ADCC?
     gold=RNC | rows=[('RNC', 99)] | err=None
OK  [1x] Quantas vitórias o Marcelo Garcia tem?
     gold=27 | rows=[(27,)] | err=None
OK  [1x] Qual a média de pontos do vencedor, considerando só lutas com placar registrado?
     gold=4.05 | rows=[(4.048913043478261,)] | err=None

Acerto: 6/6

 Marcelo Garcia tem 27 vitórias.


In [6]:
importlib.reload(src.tools)
from src.tools import query_adcc

print(query_adcc.invoke("Qual a finalização mais comum entre as mulheres?"))
time.sleep(5)
print(query_adcc.invoke("Quem venceu mais lutas por finalização no pescoço?"))

src.tools.USE_DUMMY = True               # simula ambiente do professor sem chave
print(query_adcc.invoke("qualquer coisa"))
src.tools.USE_DUMMY = False


As finalizações mais comuns entre as mulheres são **Toe hold** e **RNC**, cada uma ocorrendo 3 vezes.
SQL: SELECT submission, COUNT(*) AS n FROM lutas WHERE female = 1 AND victory_method = 'SUBMISSION' GROUP BY submission ORDER BY n DESC LIMIT 5
Marcelo Garcia venceu mais lutas por finalização no pescoço, com 19 vitórias.
SQL: SELECT winner_name, COUNT(*) AS n FROM lutas WHERE victory_method='SUBMISSION' AND submission_target='Neck' GROUP BY winner_name ORDER BY n DESC LIMIT 5
(dummy) RNC is the most common submission in ADCC (99 wins).
SQL: SELECT submission, COUNT(*) ...


## 4.4 Agente de recuperação (índice FAISS existente)

O índice da Fase 2 (`data/index/techniques/`) é reaproveitado via `load_local`, sem reindexar.
Primeiro mede-se o retrieval sozinho em casos difíceis (gíria PT, nome japonês, erro de digitação,
fora da KB, pergunta ambígua); depois o agente.


In [7]:
import pandas as pd
from src.tools import load_technique_index

index = load_technique_index()        # FAISS.load_local (lru_cache)
print("chunks no índice:", index.index.ntotal)

# (pergunta, ficha esperada | None = não existe na KB)
HARD_CASES = [
    ("armlock closed guard",                 "armlock_closed_guard.md"),   # controle (EN, termo exato)
    ("chave de braço na guarda fechada",     "armlock_closed_guard.md"),   # gíria PT
    ("juji gatame from guard",               "armlock_closed_guard.md"),   # nome japonês
    ("raspagem de tesoura",                  "scissor_sweep.md"),          # PT
    ("ude garami",                           "kimura.md"),                 # nome japonês da kimura
    ("como sair de baixo da montada",        "mount_escape_upa.md"),       # PT descritivo
    ("passar a guarda de la riva",           "dlr_guard_pass.md"),         # PT + nome próprio
    ("trinagle chok",                        "triangle_closed_guard.md"),  # erro de digitação
    ("berimbolo",                            None),                        # fora da KB
    ("como saio daí?",                       None),                        # ambígua, sem contexto
]


def similarity_retrieve(query: str, k: int = 3):
    """Retriever base: top-k por similaridade (distância L2: menor = mais parecido)."""
    return index.similarity_search_with_score(query, k=k)


def eval_retrieval(retrieve_fn, cases=HARD_CASES):
    rows = []
    for q, expected in cases:
        hits = retrieve_fn(q)
        sources = [d.metadata["source"] for d, _ in hits]
        rows.append({
            "query": q,
            "expected": expected or "—",
            "top1": f"{sources[0]} / {hits[0][0].metadata['section']}",
            "score": round(float(hits[0][1]), 3),
            "top1_ok": expected is not None and sources[0] == expected,
            "top3_ok": expected is not None and expected in sources,
        })
    df = pd.DataFrame(rows)
    in_kb = df[df.expected != "—"]
    print(f"top-1: {in_kb.top1_ok.sum()}/{len(in_kb)} | top-3: {in_kb.top3_ok.sum()}/{len(in_kb)}")
    return df


pd.set_option("display.max_colwidth", 60)
baseline = eval_retrieval(similarity_retrieve)
baseline


d:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2308.65it/s]


chunks no índice: 36
top-1: 5/8 | top-3: 6/8


,query,expected,top1,score,top1_ok,top3_ok
0,armlock closed guard,armlock_closed_guard.md,armlock_closed_guard.md / Safety,12.608,True,True
1,chave de braço na guarda fechada,armlock_closed_guard.md,armlock_closed_guard.md / Safety,12.016,True,True
2,juji gatame from guard,armlock_closed_guard.md,kimura.md / Steps,15.772,False,False
3,raspagem de tesoura,scissor_sweep.md,scissor_sweep.md / Common mistakes,13.768,True,True
4,ude garami,kimura.md,dlr_guard_pass.md / Common mistakes,14.793,False,False
5,como sair de baixo da montada,mount_escape_upa.md,dlr_guard_pass.md / Common mistakes,12.406,False,True
6,passar a guarda de la riva,dlr_guard_pass.md,dlr_guard_pass.md / Common mistakes,9.687,True,True
7,trinagle chok,triangle_closed_guard.md,triangle_closed_guard.md / Goal,15.767,True,True
8,berimbolo,—,dlr_guard_pass.md / Common mistakes,18.894,False,False
9,como saio daí?,—,dlr_guard_pass.md / Common mistakes,15.610,False,False


### Agente de recuperação

`create_agent` (LangChain 1.x, function calling nativo) com `search_technique` sobre o índice existente.
O system prompt manda traduzir a pergunta para o nome da técnica em inglês antes de buscar, citar a ficha
e admitir quando a KB não cobre o assunto.


In [8]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

RAG_SYSTEM = """You are Mestre, a BJJ instructor. Answer in Portuguese.
Knowledge base: curated English sheets on armlock/kimura/triangle from closed guard, scissor sweep,
upa mount escape and De La Riva guard pass.

Rules:
1. Before calling search_technique, rewrite the question as the standard ENGLISH technique name + position
   (e.g. 'chave de braço' -> 'armlock closed guard', 'juji gatame' -> 'armbar', 'ude garami' -> 'kimura').
2. Base the answer ONLY on the returned sections and cite the sheet name, e.g. (fonte: kimura.md).
3. If the sections do not teach what was asked, say the knowledge base does not cover it. Do not invent steps.
4. If the question is ambiguous (no position or technique), ask one clarifying question instead of searching."""

rag_agent = create_agent(llm, [search_technique], system_prompt=RAG_SYSTEM)


def run_rag_agent(question: str) -> dict:
    out = rag_agent.invoke({"messages": [("user", question)]})
    msgs = out["messages"]
    tool_inputs = [tc["args"].get("query") for m in msgs if isinstance(m, AIMessage) for tc in m.tool_calls]
    sources = sorted({line.split(" / ")[0].strip("[")
                      for m in msgs if isinstance(m, ToolMessage)
                      for line in m.content.splitlines() if line.startswith("[")})
    return {"question": question, "tool_inputs": tool_inputs, "sources": sources, "answer": msgs[-1].content}


AGENT_CASES = [
    "Como faço chave de braço na guarda fechada?",
    "Como faço um juji gatame saindo da guarda?",
    "Me ensina o ude garami",
    "Como faço o berimbolo?",
    "Como saio daí?",
]

agent_results = []
for q in AGENT_CASES:
    r = run_rag_agent(q)
    agent_results.append(r)
    print(f">>> {q}\n    busca: {r['tool_inputs']}\n    fontes: {r['sources']}\n    resposta: {r['answer'][:250]}\n")
    time.sleep(8)  # 8000 tokens/min


>>> Como faço chave de braço na guarda fechada?
    busca: ['armlock closed guard']
    fontes: ['armlock_closed_guard.md']
    resposta: **Armlock – Closed Guard (guarda fechada)**  

**Posição**  
- Você está na guarda fechada, com as pernas enroladas ao redor da cintura do oponente (gi ou no‑gi). (fonte: armlock_closed_guard.md / Position)

**Passos para executar a chave**  
1. **Co

>>> Como faço um juji gatame saindo da guarda?
    busca: ['armbar closed guard']
    fontes: ['armlock_closed_guard.md']
    resposta: **Juji‑gatame (armbar) a partir da guarda fechada**

**Passo a passo** (fonte: armlock_closed_guard.md / Steps)

1. **Controle do braço** – Segure a manga ou o pulso do adversário e puxe‑o cruzado sobre o seu peito, mantendo o cotovelo bem próximo ao

>>> Me ensina o ude garami
    busca: []
    fontes: []
    resposta: Para poder te ensinar corretamente, poderia me dizer em qual posição você quer aplicar o ude garami (kimura)? Por exemplo: guarda fechada, guarda De La 

## 4.5 Falha real nos documentos-fonte + correção com multi-query

**Diagnóstico.** `ude garami` → `dlr_guard_pass / Common mistakes` e `juji gatame` → `kimura`: as fichas só
usam o nome inglês (kimura, armlock), sem aliases japoneses/PT. Sem sobreposição de vocabulário, o embedding
cai num chunk genérico ("hub": DLR / Common mistakes foi top-1 em 5/10 queries).

**Correção.** Multi-query: um LLM pequeno gera 3 reformulações em inglês (nome padrão da técnica + posição);
busca-se com a original + as 3; une-se pelo melhor score de cada chunk. Mesmo `eval_retrieval`, mesmos casos
+ 2 inéditos (fora de qualquer prompt).


In [10]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

MQ_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You rewrite Brazilian Jiu-Jitsu questions into search queries for an ENGLISH technique "
               "knowledge base. Write 3 short English queries using the standard technique name "
               "(translate Portuguese/Japanese names) and the position. One query per line, no numbering."),
    ("human", "{question}"),
])
mq_chain = MQ_PROMPT | llm_small | StrOutputParser()


def multi_query_retrieve(query: str, k: int = 3):
    variants = [query] + [l.strip() for l in mq_chain.invoke({"question": query}).splitlines() if l.strip()][:3]
    best = {}                                            # (ficha, seção) -> (doc, menor distância)
    for v in variants:
        for doc, score in index.similarity_search_with_score(v, k=k):
            key = (doc.metadata["source"], doc.metadata["section"])
            if key not in best or score < best[key][1]:
                best[key] = (doc, score)
    multi_query_retrieve.last_variants = variants        # p/ inspeção
    return sorted(best.values(), key=lambda x: x[1])[:k]


CASES_45 = HARD_CASES + [
    ("sankaku jime",              "triangle_closed_guard.md"),  # inédito: triângulo em japonês
    ("fuga da montada com ponte", "mount_escape_upa.md"),       # inédito: upa em PT
]

print("ANTES (similarity):")
before = eval_retrieval(similarity_retrieve, CASES_45)
print("DEPOIS (multi-query):")
after = eval_retrieval(multi_query_retrieve, CASES_45)

comparison = before[["query", "expected", "top1", "top1_ok"]].merge(
    after[["query", "top1", "top1_ok"]], on="query", suffixes=("_antes", "_depois"))
comparison


ANTES (similarity):
top-1: 6/10 | top-3: 7/10
DEPOIS (multi-query):
top-1: 8/10 | top-3: 9/10


,query,expected,top1_antes,top1_ok_antes,top1_depois,top1_ok_depois
0,armlock closed guard,armlock_closed_guard.md,armlock_closed_guard.md / Safety,True,armlock_closed_guard.md / Steps,True
1,chave de braço na guarda fechada,armlock_closed_guard.md,armlock_closed_guard.md / Safety,True,armlock_closed_guard.md / Safety,True
2,juji gatame from guard,armlock_closed_guard.md,kimura.md / Steps,False,armlock_closed_guard.md / Steps,True
3,raspagem de tesoura,scissor_sweep.md,scissor_sweep.md / Common mistakes,True,scissor_sweep.md / Steps,True
4,ude garami,kimura.md,dlr_guard_pass.md / Common mistakes,False,dlr_guard_pass.md / Common mistakes,False
5,como sair de baixo da montada,mount_escape_upa.md,dlr_guard_pass.md / Common mistakes,False,dlr_guard_pass.md / Common mistakes,False
6,passar a guarda de la riva,dlr_guard_pass.md,dlr_guard_pass.md / Common mistakes,True,dlr_guard_pass.md / Common mistakes,True
7,trinagle chok,triangle_closed_guard.md,triangle_closed_guard.md / Goal,True,triangle_closed_guard.md / Defenses,True
8,berimbolo,—,dlr_guard_pass.md / Common mistakes,False,armlock_closed_guard.md / Steps,False
9,como saio daí?,—,dlr_guard_pass.md / Common mistakes,False,dlr_guard_pass.md / Common mistakes,False


In [11]:
multi_query_retrieve("ude garami")
print("variações:", multi_query_retrieve.last_variants)

# mesmo algoritmo, componente pronto (sem scores: devolve a união dos documentos)
mq_retriever = MultiQueryRetriever.from_llm(retriever=index.as_retriever(search_kwargs={"k": 3}), llm=llm_small)
docs = mq_retriever.invoke("ude garami")
print("MultiQueryRetriever:", [(d.metadata["source"], d.metadata["section"]) for d in docs])


variações: ['ude garami', 'Arm Entanglement guard', 'Arm Entanglement side control', 'Arm Entanglement mount']
MultiQueryRetriever: [('dlr_guard_pass.md', 'Common mistakes'), ('scissor_sweep.md', 'Common mistakes'), ('dlr_guard_pass.md', 'Position'), ('armlock_closed_guard.md', 'Common mistakes')]


In [12]:
mq_chain_big = MQ_PROMPT | llm | StrOutputParser()   # 120b: conhece nomes japoneses


def multi_query_rrf(query: str, k: int = 3, chain=mq_chain_big, c: int = 60):
    """Multi-query + Reciprocal Rank Fusion: chunk que aparece em várias listas sobe."""
    variants = [query] + [l.strip() for l in chain.invoke({"question": query}).splitlines() if l.strip()][:3]
    fused = {}                                       # (ficha, seção) -> [doc, menor distância, score RRF]
    for v in variants:
        for rank, (doc, dist) in enumerate(index.similarity_search_with_score(v, k=k)):
            key = (doc.metadata["source"], doc.metadata["section"])
            entry = fused.setdefault(key, [doc, dist, 0.0])
            entry[1] = min(entry[1], dist)
            entry[2] += 1 / (c + rank)
    multi_query_rrf.last_variants = variants
    ranked = sorted(fused.values(), key=lambda e: -e[2])[:k]
    return [(doc, dist) for doc, dist, _ in ranked]


print("DEPOIS v2 (multi-query 120b + RRF):")
after_v2 = eval_retrieval(multi_query_rrf, CASES_45)

comparison = comparison.merge(after_v2[["query", "top1", "top1_ok"]].rename(
    columns={"top1": "top1_v2", "top1_ok": "top1_ok_v2"}), on="query")
comparison[["query", "expected", "top1_ok_antes", "top1_ok_depois", "top1_ok_v2", "top1_v2"]]


DEPOIS v2 (multi-query 120b + RRF):
top-1: 10/10 | top-3: 10/10


,query,expected,top1_ok_antes,top1_ok_depois,top1_ok_v2,top1_v2
0,armlock closed guard,armlock_closed_guard.md,True,True,True,armlock_closed_guard.md / Safety
1,chave de braço na guarda fechada,armlock_closed_guard.md,True,True,True,armlock_closed_guard.md / Safety
2,juji gatame from guard,armlock_closed_guard.md,False,True,True,armlock_closed_guard.md / Steps
3,raspagem de tesoura,scissor_sweep.md,True,True,True,scissor_sweep.md / Defenses
4,ude garami,kimura.md,False,False,True,kimura.md / Common mistakes
5,como sair de baixo da montada,mount_escape_upa.md,False,False,True,mount_escape_upa.md / Steps
6,passar a guarda de la riva,dlr_guard_pass.md,True,True,True,dlr_guard_pass.md / Common mistakes
7,trinagle chok,triangle_closed_guard.md,True,True,True,triangle_closed_guard.md / Common mistakes
8,berimbolo,—,False,False,False,armlock_closed_guard.md / Common mistakes
9,como saio daí?,—,False,False,False,mount_escape_upa.md / Steps
